### 2024 供應鏈風險分數統計摘要
讀取 `1_llm_ready(extract risk paragraph) .ipynb` 產出的 article-level / snippet-level CSV，計算分數分布、零分比例、每篇 snippet 平均數量、matched_bigram 頻率等統計，用來檢查跑批結果是否合理。

> 執行本 notebook 前，請先確認 `1_llm_ready...` 已經跑完 cell 6，產生了下面兩個 2024 年的輸出檔。

In [ ]:
import pandas as pd

# 沿用 1_llm_ready(extract risk paragraph) .ipynb 中定義的輸出路徑
article_csv = r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2024\article_level_scores_2024.csv'
snippet_csv = r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2024\llm_ready_data_2024_context50_snippetlevel.csv'

df_article = pd.read_csv(article_csv)
df_snippet = pd.read_csv(snippet_csv)

print(f"article-level 筆數: {len(df_article)}")
print(f"snippet-level 筆數: {len(df_snippet)}")

#### 1. 分數分布 + 零分比例（來源：article-level）

In [ ]:
# 分數描述性統計
print("=== sc_risk_score 描述性統計 ===")
print(df_article['sc_risk_score'].describe())

# 分數分桶分布（區間可依實際分數量級調整）
bins = [-0.0001, 0, 0.01, 0.05, 0.1, 0.2, 0.5, float('inf')]
labels = ['0', '(0, 0.01]', '(0.01, 0.05]', '(0.05, 0.1]', '(0.1, 0.2]', '(0.2, 0.5]', '(0.5, +]']
score_bucket = pd.cut(df_article['sc_risk_score'], bins=bins, labels=labels)
bucket_dist = score_bucket.value_counts().sort_index()
bucket_dist_pct = (bucket_dist / len(df_article) * 100).round(2)

print("\n=== sc_risk_score 分桶分布 ===")
print(pd.DataFrame({'文章數': bucket_dist, '佔比(%)': bucket_dist_pct}))

# 零分比例
zero_count = (df_article['sc_risk_score'] == 0).sum()
zero_ratio = zero_count / len(df_article)
print(f"\n=== 零分比例 ===")
print(f"零分文章數: {zero_count} / 總文章數: {len(df_article)} = {zero_ratio:.2%}")

#### 2. 每篇 snippet 平均數量（來源：snippet-level + article-level）

In [ ]:
snippet_counts = df_snippet.groupby('file_name').size()

print("=== 每篇 snippet 數量描述性統計（僅計有 snippet 的文章）===")
print(snippet_counts.describe())

avg_among_snippet_articles = snippet_counts.mean()
avg_among_all_articles = snippet_counts.sum() / len(df_article)

print(f"\n有 snippet 的文章平均 snippet 數: {avg_among_snippet_articles:.2f}"
      f"（共 {len(snippet_counts)} 篇有 snippet 的文章）")
print(f"全部文章（含零分/無 snippet）平均 snippet 數: {avg_among_all_articles:.2f}"
      f"（共 {len(df_article)} 篇文章）")

#### 3. matched_bigram 頻率（來源：snippet-level）

In [ ]:
bigram_freq = df_snippet['matched_bigram'].value_counts()

print(f"unique matched_bigram 數量: {bigram_freq.shape[0]}")
print(f"matched_bigram 總出現次數: {bigram_freq.sum()}")
print("\n=== matched_bigram 出現次數前 20 名 ===")
print(bigram_freq.head(20))

#### 4. 其他輔助欄位（視需要擴充）

In [ ]:
print("=== event_type 分布（article-level） ===")
print(df_article['event_type'].value_counts())

print("\n=== total_bigrams 描述性統計（article-level） ===")
print(df_article['total_bigrams'].describe())